# 05 · Tablero interactivo — Adopción de IA en empresas

Este cuaderno lleva el LAB06 al conjunto de datos del equipo. El tablero se construye primero en Plotly y la misma pregunta se ofrece como aplicación de Streamlit en `src/tablero.py`.


## Audiencia, decisión y preguntas

**Audiencia:** el equipo que sigue la adopción de IA en empresas.

**Decisión recurrente:** decidir qué regiones e industrias conviene revisar con más detalle, manteniendo visible la evolución del período completo seleccionado.

**Objetivo del tablero:** describir cómo varía la tasa media de adopción de IA (`ai_adoption_rate`, porcentaje de 0 a 100) entre 2023 y 2026. No se fijó un valor meta porque la fuente no propone uno.

Las tres vistas contestan estas preguntas:

1. ¿Cómo cambió la tasa media de adopción por año?
2. ¿Qué regiones quedan por encima o por debajo del promedio del recorte?
3. ¿Qué industrias quedan por encima o por debajo del promedio del recorte?

La jerarquía pone primero los indicadores de resumen, después la tendencia temporal y al final las comparaciones por región e industria. Los filtros de región, industria, tamaño y período son globales: todas las vistas recalculan sobre la misma selección.


In [1]:
from pathlib import Path
import sys
import pandas as pd
import ipywidgets as widgets
from IPython.display import display

RAIZ = next(p for p in [Path.cwd().resolve(), *Path.cwd().resolve().parents]
            if (p / 'proyecto_equipo' / 'datos').is_dir())
PROYECTO = RAIZ / 'proyecto_equipo'
sys.path.insert(0, str(PROYECTO / 'src'))
from visualizaciones_ai import (REGIONES, INDUSTRIAS, TAMANOS,
                                preparar_datos, filtrar, crear_tablero)

RUTA = PROYECTO / 'datos' / 'derivados' / 'ai_company_adoption_limpio.parquet'
datos = preparar_datos(pd.read_parquet(RUTA))
print(f'{len(datos):,} filas empresa-trimestre · {datos.company_id.nunique():,} empresas · '
      f'{datos.survey_year.min()}–{datos.survey_year.max()}')
print('Se usa una media simple: el dataset no publica ponderaciones.')

150,000 filas empresa-trimestre · 10,000 empresas · 2023–2026
Se usa una media simple: el dataset no publica ponderaciones.


El tablero se genera en una sola figura Plotly con tres paneles; al pasar el cursor se leen los valores y tamaños de muestra, y la figura permite acercar y explorar los datos. La aplicación Streamlit aplica filtros globales a las tres vistas a la vez.

Las observaciones trimestrales se repiten por empresa y la fuente no documenta un diseño muestral ni ponderaciones; por eso el tablero no presenta intervalos de confianza ni interpreta las diferencias como causales. Las métricas describen esta base, no toda la población de empresas.


In [2]:
figura = crear_tablero(datos)
figura.show()

## Ejercicios de diseño

**Cambio de audiencia (adaptado al dataset):** si el tablero lo mira un equipo ejecutivo que decide dónde profundizar la estrategia de adopción, la frase es «cuando el equipo ejecutivo mire este tablero, va a decidir qué regiones e industrias requieren seguimiento». Los tres indicadores de resumen quedan arriba; la evolución muestra el contexto temporal y las comparaciones indican dónde investigar. El tamaño de empresa sigue disponible como filtro.

**Prueba de densidad:** si se quita la evolución anual, se pierde el contexto temporal; si se quita la comparación regional, ya no se ven diferencias geográficas; si se quita la comparación industrial, se pierde la lectura por sector. Las tres vistas contestan preguntas distintas y no son redundantes. Se conserva el tablero central de tres vistas que pide el laboratorio.


### Ejercicio del conjunto del equipo: tablero de cuatro vistas

La guía también pide probar cuatro vistas con el dataset propio. Esta segunda figura agrega el tamaño de empresa; cada panel explicita la pregunta que responde. Es un ejercicio de densidad separado del tablero principal de tres vistas: la aplicación Streamlit queda con tres para mantener la jerarquía de la entrega.

- ¿Cómo cambia la adopción media por año?
- ¿Qué regiones están por encima o debajo de la media del recorte?
- ¿Qué industrias están por encima o debajo de la media del recorte?
- ¿Cómo varía la adopción media por tamaño de empresa?


In [3]:
figura_cuatro_vistas = crear_tablero(datos, incluir_tamano=True)
figura_cuatro_vistas.show()

## Lectura, ética y límites

El titular se calcula con el primer y último año del período seleccionado; si se elige un solo año, informa el nivel de ese año. Las barras empiezan en cero y usan una escala común (0–100 %). Al pasar el cursor se leen el número de empresas distintas y las filas empresa-trimestre incluidas.

**Fuente:** mohankrishnathalla, *Global AI Adoption & Workforce Impact Dataset*, Kaggle; archivo `ai_company_adoption.csv`, consultado en octubre de 2026. **Unidad:** empresa-trimestre. **Período:** 2023–2026. **Transformación:** media aritmética simple, sin ponderar. La fuente no explica cómo se generaron los datos ni publica un diseño muestral verificable. Las filas se repiten por empresa, por lo que no se presentan los conteos como observaciones independientes; no se muestran intervalos de confianza ni se extrapolan los resultados. Las asociaciones no implican causalidad.

La aplicación persistente equivalente se inicia desde la raíz del repositorio con `streamlit run proyecto_equipo/src/tablero.py`.
